In [1]:
import pyodbc
import os
import pandas as pd
from datetime import datetime, date
import gspread
from google.oauth2.service_account import Credentials
from gspread_dataframe import get_as_dataframe
from dotenv import load_dotenv , find_dotenv # to load .env files
# from google.cloud import storage

# client = storage.Client()
load_dotenv()

True

In [2]:
prod_servr = os.environ.get("DB2_HOST")
prod_user = os.environ.get("DB2_USER")
prod_pass = os.environ.get("DB2_PASS")
prod_db = os.environ.get("DB2_NAME") 

conn = pyodbc.connect(
            f"Driver={{ODBC Driver 17 for SQL Server}};"
            f"Server={prod_servr};"  # Production server
            f"Database={prod_db};"  # Production database
            f"UID={prod_user};"
            f"PWD={prod_pass};"
        )

prod_cursor = conn.cursor()

prod_cursor.fast_executemany = True

In [3]:
# Path to your downloaded key
service_acc_key = os.getenv("gcp_bot_key")

# Define the required scopes
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]

# Authorize
creds = Credentials.from_service_account_file(service_acc_key, scopes=SCOPES)
gc = gspread.authorize(creds)

# Initial Fetching Doc

In [8]:
order_booking = 'https://docs.google.com/spreadsheets/d/19RvB03N1Az6ORh4-UPk4JPLMbWnTv6A5pfpwbgcPVBc/edit?usp=sharing'
order_booking_sheet = gc.open_by_url(order_booking)
order_booking_df = pd.DataFrame(order_booking_sheet.worksheet('Clean Order').get_all_records())
order_booking_df[:10]

,Id,Select_route,route_id,Select_Day,Day_ID,Type_RRP_Mobile_no,Fetched_RRP,kiosk_code,lat_code,long_code,...,Type_Quantity,Product_Id,Product_Price,userCode,Damage_return_product,Changed_date,orderKey,OrderId,orderkey2,Order_complete
0,2,Bhangore 2 Route 1 (ID:00700),700,Wednesday,4,9093350424,Bhangore 2 Ph.no:9093350424 (ID:112407),112407,22.59455,88.396534,...,60,3620,7.00,70872,Damage,,70011240746092124,,700112407460921243620,done
1,3,Bhangore 2 Route 1 (ID:00700),700,Wednesday,4,9093350424,Bhangore 2 Ph.no:9093350424 (ID:112407),112407,22.59455,88.396534,...,120,3622,3.50,70872,Damage,,70011240746092124,,700112407460921243622,done
2,4,Bhangore 2 Route 1 (ID:00700),700,Wednesday,4,9093350424,Bhangore 2 Ph.no:9093350424 (ID:112407),112407,22.59455,88.396534,...,120,3623,3.33,70872,Damage,,70011240746092124,,700112407460921243623,done
3,5,Bhangore 2 Route 1 (ID:00700),700,Wednesday,4,9093350424,Bhangore 2 Ph.no:9093350424 (ID:112407),112407,22.59455,88.396534,...,120,3624,3.50,70872,Damage,,70011240746092124,,700112407460921243624,done
4,6,Bhangore 2 Route 1 (ID:00700),700,Wednesday,4,9093350424,Bhangore 2 Ph.no:9093350424 (ID:112407),112407,22.59455,88.396534,...,120,3625,3.50,70872,Damage,,70011240746092124,,700112407460921243625,done
5,7,Bhangore 2 Route 1 (ID:00700),700,Wednesday,4,9093350424,Bhangore 2 Ph.no:9093350424 (ID:112407),112407,22.59455,88.396534,...,144,3626,5.83,70872,Damage,,70011240746092124,,700112407460921243626,done
6,8,Bhangore 2 Route 1 (ID:00700),700,Wednesday,4,9093350424,Bhangore 2 Ph.no:9093350424 (ID:112407),112407,22.59455,88.396534,...,120,3627,3.50,70872,Damage,,70011240746092124,,700112407460921243627,done
7,9,Bhangore 2 Route 1 (ID:00700),700,Wednesday,4,9093350424,Bhangore 2 Ph.no:9093350424 (ID:112407),112407,22.59455,88.396534,...,120,3628,3.50,70872,Damage,,70011240746092124,,700112407460921243628,done
8,10,Bhangore 2 Route 1 (ID:00700),700,Wednesday,4,9093350424,Bhangore 2 Ph.no:9093350424 (ID:112407),112407,22.59455,88.396534,...,2,3901,600.00,70872,,11-03-2026,70011240746092124,38548,700112407460921243901,done
9,11,Bhangore 2 Route 1 (ID:00700),700,Wednesday,4,9093350424,Bhangore 2 Ph.no:9093350424 (ID:112407),112407,22.59455,88.396534,...,2,4394,42.00,70872,Damage,,70011240746092124,,700112407460921244394,done


# Transformation

In [2]:

order_booking_df['Select_Delivery_Date'] = pd.to_datetime(
    order_booking_df['Select_Delivery_Date'].str.strip(),
    format='%d-%m-%Y'
)

order_booking_df['Changed_date'] = pd.to_datetime(
    order_booking_df['Changed_date'].str.strip(),
    format='%d-%m-%Y'
)

order_booking_df['lat_code'] = pd.to_numeric(
    order_booking_df['lat_code'].astype(str).str.strip(),
    errors='coerce'
)

order_booking_df['long_code'] = pd.to_numeric(
    order_booking_df['long_code'].astype(str).str.strip(),
    errors='coerce'
)

order_booking_df['Product_Price'] = pd.to_numeric(
    order_booking_df['Product_Price'].astype(str).str.strip(),
    errors='coerce'
)

order_booking_df['route_id_str'] = order_booking_df['route_id'].apply(lambda x: f"{x:05d}")

# order_booking_df.info()


NameError: name 'order_booking_df' is not defined

# Filtering Order

In [45]:
rrp = 104664
route = '01606' 
date1 = '12-03-2026'
date1 = pd.to_datetime(date1, dayfirst=True)
user = 69869
officeid = 117


order_booking_df2 = order_booking_df[(order_booking_df['kiosk_code'] == rrp) & (order_booking_df['route_id_str'] == route) & (order_booking_df['Changed_date'] == date1) & (order_booking_df['userCode'] == user)].copy()

order_booking_df2['Select_Delivery_Date'] = order_booking_df2['Changed_date']

order_booking_df2.values


array([[141, 'Route 1 (ID:01606)', 1606, 'Thursday', 5, 7635843790,
        'Maslam Ph.no:7635843790 (ID:104664)', 104664, 26.0091857,
        91.0189768, Timestamp('2026-03-12 00:00:00'),
        'Goalpara (ID:117)', 117, 'CUP CAKES', 300, 2607, 5.9, 69869, '',
        Timestamp('2026-03-12 00:00:00'), 160610466446093117, '',
        1606104664460931172607, 'not', '01606'],
       [142, 'Route 1 (ID:01606)', 1606, 'Thursday', 5, 7635843790,
        'Maslam Ph.no:7635843790 (ID:104664)', 104664, 26.0091857,
        91.0189768, Timestamp('2026-03-12 00:00:00'),
        'Goalpara (ID:117)', 117,
        'PATANJALI LIME FRESH BODY CLEANSER 75 GM 144x25', 10, 6575,
        18.01, 69869, '', Timestamp('2026-03-12 00:00:00'),
        160610466446093117, '', 1606104664460931176575, 'not', '01606']],
      dtype=object)

In [39]:
len(order_booking_df2)

12

# Check

In [10]:
len(order_booking_df['route_id'].unique()) , len(order_booking_df['route_id_str'].unique())

(24, 24)

In [9]:
prod_cursor.execute(
    """
    EXEC drishteeDhavak..sp_scm__rrp_order_booking_insert
    ?,?,?,?,?,?
    """,
    86122, 593, '2026-03-12 00:00:00', 68779, 22.48320007, 82.5750885
)

result = prod_cursor.fetchone()

messagecode = result[0]
message = result[1]
order_id = result[2]

order_id

38548

In [1]:
prod_cursor.execute("""
    EXEC drishtee_group_mis..usp_check_dhavak_stock
        ?,?,?
    """,
    3620,124,'2026-03-11'
)

stock = prod_cursor.fetchone()

if stock is None:
    print(f"Skipping Product (No stock record found)")
    pass
else:
    product_id = stock[0]
    product_name = stock[1]
    available_quantity = stock[2]
    stock_available = stock[5]

    print(f"Product {product_id} is available with stock: {stock_available}")

NameError: name 'prod_cursor' is not defined

In [ ]:
# orderbooking
print('kiosk_code ','route_id ','Select_Delivery_Date ','Office_Id ','lat_code ','long_code ','userCode ')

# Import Dhavak
print('rrp_u ','route_id ','Select_Delivery_Date ','Office_Id ','userCode ',)

# Product Issue
print('userCode ', 'route_id ','Select_Delivery_Date ', 'Office_Id')

# Sale Entry
print('kiosk_code','route_id', 'Select_Delivery_Date','userCode')

kiosk_code  route_id  Select_Delivery_Date  Office_Id  lat_code  long_code  userCode 
rrp_u  route_id  Select_Delivery_Date  userCode  Office_Id 
userCode  Select_Delivery_Date  route_id  Office_Id
kiosk_code route_id Select_Delivery_Date userCode


# Orderbooking Key

In [46]:
grouped_orders = order_booking_df2.groupby([
    "kiosk_code",
    "route_id_str",
    "Select_Delivery_Date",
    "Office_Id",
    "lat_code",
    "long_code",
    "userCode"
])

print('kiosk_code ',
    'route_id_str ',
    'Select_Delivery_Date ',
    'Office_Id ',
    'lat_code ',
    'long_code ',
    'userCode ')

for i in grouped_orders.groups:
    print(i)

kiosk_code  route_id_str  Select_Delivery_Date  Office_Id  lat_code  long_code  userCode 
(104664, '01606', Timestamp('2026-03-12 00:00:00'), 117, 26.0091857, 91.0189768, 69869)


In [47]:
for key, order_df in grouped_orders:

    kiosk_code, route_id_str, delivery_date, office_id, lat, long, userCode = key

    print(f"Order for Kiosk kiosk_code, route_id, delivery_date, office_id, lat, long, userCode \n{kiosk_code, route_id_str, delivery_date, office_id, lat, long, userCode}")

    for _, row in order_df.iterrows():

        product_id = row['Product_Id']
        qty = pd.to_numeric(row['Type_Quantity'], errors='coerce')

        prod_cursor.execute("""
        EXEC drishtee_group_mis..usp_check_dhavak_stock
            ?,?,?
        """,
        int(product_id),
        int(office_id),
        pd.to_datetime(delivery_date).to_pydatetime()
        )

        stock = prod_cursor.fetchone()
        print(stock)



Order for Kiosk kiosk_code, route_id, delivery_date, office_id, lat, long, userCode 
(104664, '01606', Timestamp('2026-03-12 00:00:00'), 117, 26.0091857, 91.0189768, 69869)
(2607, 'CUP CAKES', 300, 0, 0, 300)
(6575, 'PATANJALI LIME FRESH BODY CLEANSER 75 GM 144x25', 10, 0, 0, 10)


# OrderbookingInsert

In [48]:
for key, order_df in grouped_orders:

    kiosk_code, route_id_str, delivery_date, office_id, lat, long, userCode = key

    print(f"Creating Order for Kiosk kiosk_code, route_id, delivery_date, office_id, lat, long, userCode \n{kiosk_code, route_id_str, delivery_date, office_id, lat, long, userCode}")

    # STEP 1 : CREATE ORDER HEADER
    prod_cursor.execute("""
    EXEC drishteeDhavak..sp_scm__rrp_order_booking_insert
        ?,?,?,?,?,?
    """,
    int(kiosk_code),
    str(route_id_str),
    pd.to_datetime(delivery_date).to_pydatetime(),
    int(userCode),
    float(long),
    float(lat)
    )

    result = prod_cursor.fetchone()

    messagecode = result[0]
    message = result[1]
    order_id = result[2]
    
    print("Order Created:", order_id)
    print("Message Code:", messagecode, "Message:", message, "Order ID:", order_id)

    # STEP 2 : PROCESS PRODUCTS
    for _, row in order_df.iterrows():

        product_id = row['Product_Id']
        qty = pd.to_numeric(row['Type_Quantity'], errors='coerce')

        prod_cursor.execute("""
        EXEC drishtee_group_mis..usp_check_dhavak_stock
            ?,?,?
        """,
        int(product_id),
        int(office_id),
        pd.to_datetime(delivery_date).to_pydatetime()
        )

        stock = prod_cursor.fetchone()

        if stock is None:
            print(f"Skipping Product {product_id} (No stock record found)")
            prod_cursor.commit()
            conn.commit()
            continue
        
        product_id = stock[0]
        product_name = stock[1]
        available_quantity = stock[2]
        stock_available = stock[5]

        print(f'Product Name:{product_id}')

        if float(stock_available) >= qty:

            print(f"Inserting Product {product_id} for orderId {order_id}")

            prod_cursor.execute("""
            EXEC drishteeDhavak..sp_scm__rrp_order_booking_detail_insert
                ?,?,?,?
            """,
            int(order_id),
            int(userCode),
            int(product_id),
            int(qty)
            )

            prod_cursor.commit()
            conn.commit()
            
        else:
            print(f"Skipping Product {product_id} ,for route_id:{route_id_str}, kiosk_code:{kiosk_code}, lat:{lat}, long:{long}, delivery_date:{delivery_date}, office_id:{office_id}, userCode:{userCode},  OrderID:{order_id}")
    
    prod_cursor.commit()
    conn.commit()

print("Orderbooking Data inserted successfully...!")

Creating Order for Kiosk kiosk_code, route_id, delivery_date, office_id, lat, long, userCode 
(104664, '01606', Timestamp('2026-03-12 00:00:00'), 117, 26.0091857, 91.0189768, 69869)
Order Created: 38590
Message Code: 1 Message: Data submitted succesfully. Order ID: 38590
Product Name:2607
Inserting Product 2607 for orderId 38590
Product Name:6575
Inserting Product 6575 for orderId 38590
Orderbooking Data inserted successfully...!


# Import to Dhavak Data

In [5]:
order_booking = 'https://docs.google.com/spreadsheets/d/19RvB03N1Az6ORh4-UPk4JPLMbWnTv6A5pfpwbgcPVBc/edit?usp=sharing'
order_booking_sheet = gc.open_by_url(order_booking)
order_booking_df = pd.DataFrame(order_booking_sheet.worksheet('Clean Order').get_all_records())

order_booking_df['lat_code'] = pd.to_numeric(
    order_booking_df['lat_code'].astype(str).str.strip(),
    errors='coerce'
)

order_booking_df['long_code'] = pd.to_numeric(
    order_booking_df['long_code'].astype(str).str.strip(),
    errors='coerce'
)

order_booking_df['Product_Price'] = pd.to_numeric(
    order_booking_df['Product_Price'].astype(str).str.strip(),
    errors='coerce'
)

clean_order_booking_df = order_booking_df[(order_booking_df['Import_Flag'] == 'Yes') & (order_booking_df['Product_Price'] > 0)].copy()
# clean_order_booking_df

In [6]:
clean_order_booking_df['Select_Delivery_Date'] = pd.to_datetime(
    clean_order_booking_df['Select_Delivery_Date'].str.strip(),
    format='%d-%m-%Y'
)

clean_order_booking_df['lat_code'] = pd.to_numeric(
    clean_order_booking_df['lat_code'].astype(str).str.strip(),
    errors='coerce'
)

clean_order_booking_df['long_code'] = pd.to_numeric(
    clean_order_booking_df['long_code'].astype(str).str.strip(),
    errors='coerce'
)

clean_order_booking_df['Product_Price'] = pd.to_numeric(
    clean_order_booking_df['Product_Price'].astype(str).str.strip(),
    errors='coerce'
)

clean_order_booking_df['rrp_u'] = (
    'D~' + clean_order_booking_df['route_id']
    .astype(str)
    .str.replace(r'[\n\r]', '', regex=True)
    .str.strip()
)

clean_order_booking_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 60 entries, 8 to 139
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Id                    60 non-null     int64         
 1   Import_Flag           60 non-null     object        
 2   Select_route          60 non-null     object        
 3   route_id              60 non-null     int64         
 4   Select_Day            60 non-null     object        
 5   Day_ID                60 non-null     int64         
 6   Type_RRP_Mobile_no    60 non-null     int64         
 7   Fetched_RRP           60 non-null     object        
 8   kiosk_code            60 non-null     int64         
 9   lat_code              60 non-null     float64       
 10  long_code             60 non-null     float64       
 11  Select_Delivery_Date  60 non-null     datetime64[ns]
 12  Select_Office         60 non-null     object        
 13  Office_Id             60 n

In [ ]:
clean_order_booking_df['rrp_u'].unique()

array(['D~700', 'D~797', 'D~598', 'D~3', 'D~1509', 'D~1389', 'D~1313',
       'D~593', 'D~683', 'D~1978', 'D~760', 'D~1901', 'D~1161', 'D~810',
       'D~705'], dtype=object)

# Import Dhavak Key

In [ ]:
order_booking_df2['rrp_u'] = (
    'D~' + order_booking_df2['route_id_str']
    .astype(str)
    .str.replace(r'[\n\r]', '', regex=True)
    .str.strip()
)

clean_order_booking_df = order_booking_df2.copy()
clean_order_booking_df

,Id,Select_route,route_id,Select_Day,Day_ID,Type_RRP_Mobile_no,Fetched_RRP,kiosk_code,lat_code,long_code,...,Product_Price,userCode,Damage_return_product,Changed_date,orderKey,OrderId,orderkey2,Order_complete,route_id_str,rrp_u
152,141,Route 1 (ID:01606),1606,Thursday,5,7635843790,Maslam Ph.no:7635843790 (ID:104664),104664,26.009186,91.018977,...,5.90,69869,,2026-03-12,160610466446093117,,1606104664460931172607,not,01606,D~01606
153,142,Route 1 (ID:01606),1606,Thursday,5,7635843790,Maslam Ph.no:7635843790 (ID:104664),104664,26.009186,91.018977,...,18.01,69869,,2026-03-12,160610466446093117,,1606104664460931176575,not,01606,D~01606


In [ ]:
grouped_orders = order_booking_df2.groupby([
    'rrp_u',
    'route_id_str',
    'Select_Delivery_Date',
    'userCode',
    'Office_Id'
])

print('rrp_u ',
    'route_id_str ',
    'Select_Delivery_Date ',
    'userCode ',
    'Office_Id ')

for i in grouped_orders.groups:
    print(i)

rrp_u  route_id_str  Select_Delivery_Date  userCode  Office_Id 
('D~01606', '01606', Timestamp('2026-03-12 00:00:00'), 69869, 117)


# Procedure Wise

In [ ]:
prod_cursor.execute("""
EXEC drishtee_group_mis..OrderBooking_Insert_proc ?,?,?,?,?
""",
'D~1161', 705, '2026-03-12', 68779, 60
# int(kiosk_code), int(route_id), delivery_date, int(userCode), int(office_id)
)

result = prod_cursor.fetchone()
order_id = result[0]

order_id

20260306001280

In [51]:
prod_cursor.execute(f"""
EXEC drishtee_group_mis..select_orderfromdhavak_with_stock_proc ?,?,?
""",
'00598', '2026-03-11',116  
# int(route_id), pd.to_datetime(delivery_date).to_pydatetime(), int(office_id)
)

print(int('00598'))

columns = [col[0] for col in prod_cursor.description]
stock_rows = [tuple(i) for i in prod_cursor.fetchall()]

stock_df = pd.DataFrame.from_records(stock_rows, columns=columns)
stock_df

598


,product_id,quantity,route_id,office_id,delivery_date,tax,unit_price,product_name,peti_quantity,tax_amount,available_quanitty,last_purchase,brand_name
0,4071,14362,00598,116,2026-03-11,5.00,0.82,Agarbatti Packet Lalita Brand,NaN,588.84200000,14362,0.00,723
1,4075,14000,00598,116,2026-03-11,18.00,0.46,Lalita Brand Label for Phenyl,1000.0,1159.20000000,14000,0.46,723


In [18]:
stock_df1 = stock_df[~stock_df['unit_price'].isna()].copy()
stock_df1

,product_id,quantity,route_id,office_id,delivery_date,tax,unit_price,product_name,peti_quantity,tax_amount,available_quanitty,last_purchase,brand_name
0,3820,60,1389,60,2026-03-12,5.00,95.00,Oyster spwan Seeds,400,285.00000000,650,95.0,663


In [59]:
clean_order_booking_df.columns , stock_df1.columns

(Index(['Id', 'Import_Flag', 'Select_route', 'route_id', 'Select_Day', 'Day_ID',
        'Type_RRP_Mobile_no', 'Fetched_RRP', 'kiosk_code', 'lat_code',
        'long_code', 'Select_Delivery_Date', 'Select_Office', 'Office_Id',
        'Select_Product', 'Type_Quantity', 'Product_Id', 'Product_Price',
        'userCode', 'orderKey1', 'Full_order_Flag', 'orderKey2', 'Order_id',
        'orderKey3', 'orderKey4', 'Amount', 'Product_key', 'route_id_str',
        'rrp_u'],
       dtype='object'),
 Index(['product_id', 'quantity', 'route_id', 'office_id', 'delivery_date',
        'tax', 'unit_price', 'product_name', 'peti_quantity', 'tax_amount',
        'available_quanitty', 'last_purchase', 'brand_name'],
       dtype='object'))

In [60]:
# clean_order_booking_df['route_id'] = clean_order_booking_df['route_id'].astype(int)

# stock_df1['route_id'] = stock_df1['route_id'].astype(int)

merged_df = pd.merge(
    clean_order_booking_df,
    stock_df1,
    left_on=['Product_Id', 'route_id_str', 'Select_Delivery_Date', 'Office_Id'],
    right_on=['product_id', 'route_id', 'delivery_date', 'office_id'],
    how='inner'
)

merged_df = merged_df[['product_id', 'quantity', 'route_id_y', 'office_id', 'delivery_date',
       'tax', 'unit_price', 'product_name', 'peti_quantity', 'tax_amount',
       'available_quanitty', 'last_purchase', 'brand_name']].drop_duplicates().copy()
merged_df

,product_id,quantity,route_id_y,office_id,delivery_date,tax,unit_price,product_name,peti_quantity,tax_amount,available_quanitty,last_purchase,brand_name
0,4071,14362,00598,116,2026-03-11,5.00,0.82,Agarbatti Packet Lalita Brand,NaN,588.84200000,14362,0.00,723
1,4075,14000,00598,116,2026-03-11,18.00,0.46,Lalita Brand Label for Phenyl,1000.0,1159.20000000,14000,0.46,723


In [21]:
for row in merged_df.itertuples():

    # print(f"Inserting Product {row.product_id} for orderId {order_id}")

    # prod_cursor.execute("""
    # EXEC drishtee_group_mis..OrderBookingDetail_Insert_proc
    #     ?,?,?,?,?,?,?,?,?,?,?,?,?,?
    # """,
    print(
    int(order_id),
    int(row.product_id),
    float(row.quantity),
    float(row.unit_price),
    int(userCode),
    float(row.tax),
    float(row.tax_amount),
    int(office_id),
    float(row.peti_quantity),
    float(row.unit_price),
    float(row.unit_price),
    0,
    'F',
    'N'
    )

202603012803 3331 28.0 23.82 68812 5.0 33.348 128 28.0 23.82 23.82 0 F N


# Import to Dhavak Insertion

In [2]:
for keys, order_df in grouped_orders:
    rrp_u, route_id_str, delivery_date, userCode, office_id = keys

    print(f"\nProcessing Order → RRP:{rrp_u} route:{route_id_str} delivery_date:{delivery_date} userCode:{userCode} office_id:{office_id}")

    delivery_date = pd.to_datetime(delivery_date).to_pydatetime()

    prod_cursor.execute("""
    EXEC drishtee_group_mis..OrderBooking_Insert_proc ?,?,?,?,?
    """,
    str(rrp_u),
    str(route_id_str),
    delivery_date,
    int(userCode),
    int(office_id)
    )

    result = prod_cursor.fetchone()
    order_id = result[0]
    print("Order ID:", order_id)

    if result is None:
        print("Order creation failed\n")
        continue

    print(f"Fetching stock for route_id:{route_id_str}, delivery_date:{delivery_date}, office_id:{office_id}")

    prod_cursor.fast_executemany = True

    prod_cursor.execute("""
    EXEC drishtee_group_mis..select_orderfromdhavak_with_stock_proc ?,?,?
    """,
    str(route_id_str),
    delivery_date,
    int(office_id)
    )
    columns = [col[0] for col in prod_cursor.description]
    stock_rows = [tuple(i) for i in prod_cursor.fetchall()]
    stock_df = pd.DataFrame.from_records(stock_rows, columns=columns)

    stock_df1 = stock_df[~stock_df['unit_price'].isna()].copy()

    if not stock_df1.shape[0]:
        print("No stock returned\n")
        continue
    

    # clean_order_booking_df['route_id'] = clean_order_booking_df['route_id'].astype(int)
    # stock_df1['route_id'] = stock_df1['route_id'].astype(int)
    merged_df = pd.merge(
    clean_order_booking_df,
    stock_df1,
    left_on=['Product_Id', 'route_id_str', 'Select_Delivery_Date', 'Office_Id'],
    right_on=['product_id', 'route_id', 'delivery_date', 'office_id'],
    how='inner'
    )

    merged_df1 = merged_df[['product_id', 'quantity', 'route_id_y', 'office_id', 'delivery_date',
        'tax', 'unit_price', 'product_name', 'peti_quantity', 'tax_amount',
        'available_quanitty', 'last_purchase', 'brand_name']].drop_duplicates().copy()

    for row in merged_df1.itertuples():

        print(f"Inserting Product {row.product_id} for orderId {order_id}")

        prod_cursor.execute("""
        EXEC drishtee_group_mis..OrderBookingDetail_Insert_proc
            ?,?,?,?,?,?,?,?,?,?,?,?,?,?
        """,
        int(order_id),
        int(row.product_id),
        float(row.quantity or 0),
        float(row.unit_price or 0),
        int(userCode),
        float(row.tax or 0),
        float(row.tax_amount or 0),
        int(office_id),
        float(row.peti_quantity or 0),
        float(row.unit_price or 0),
        float(row.unit_price or 0),
        float(0),
        'F',
        'N'
        )

    prod_cursor.commit()
    conn.commit()

    print(f"Order Completed for: RRP={rrp_u}, route_id={route_id_str}, delivery_date={delivery_date}, office_id={office_id}, userCode={userCode}, order_id={order_id}")

print(f"\nOrderbooking Data Inserted...!")


NameError: name 'grouped_orders' is not defined

# Issue to Executive Key

In [52]:
grouped_orders = clean_order_booking_df.groupby(['userCode', 'Select_Delivery_Date','route_id_str', 'Office_Id'])

print('userCode ', 'Select_Delivery_Date ','route_id_str ', 'Office_Id')
for i in grouped_orders.groups:
    print(i)

userCode  Select_Delivery_Date  route_id_str  Office_Id
(69869, Timestamp('2026-03-12 00:00:00'), '01606', 117)


# Issue to Executive

In [ ]:
for keys, order_df in grouped_orders:
    userCode, Delivery_Date,route_id_str, Office_Id = keys

    print(f"\nIssuing Order :\nuserCode:{userCode} Delivery_Date:{Delivery_Date} route_id:{route_id_str} Office_Id:{Office_Id}")

    delivery_date = pd.to_datetime(Delivery_Date).to_pydatetime()

    prod_cursor.execute("""
    SELECT CONCAT(LTRIM(TRIM(UserFirstName)),' ',LTRIM(TRIM(UserMiddleName)),' ',LTRIM(TRIM(UserLastName))) transporter_name
    FROM drishteeDhavak..usermaster WHERE UserDrishteeID = ?
    """,
    int(userCode)
    )
    columns = [col[0] for col in prod_cursor.description]
    rows = [tuple(i) for i in prod_cursor.fetchall()]
    dhavak = pd.DataFrame.from_records(rows, columns=columns)

    print(f"Issue For Dhavak: {dhavak.transporter_name[0]}\n")
    
    prod_cursor.execute("""
     drishtee_group_mis..ProductIssueRRA_insert_update ?,?,?,?,?,?,?,?,?,?
    """,
    int(userCode),                                      
    delivery_date,                                      
    'EMP967',                                      
    str(route_id_str),                                    
    'I',                                      
    int(Office_Id),                  
    str(dhavak.transporter_name[0]),                   
    '1234',                    
    10,                    
    1
    )
    
#     prod_cursor.execute("""
#     drishtee_group_mis..ProductIssueRRA_insert_update_1 ?,?,?,?,?,?,?,?,?,?
#     """,
#         int(userCode),                                      
#         delivery_date,                                      
#         'EMP967',                                      
#         str(route_id_str),                                    
#         'I',                                      
#         int(Office_Id),                  
#         str(dhavak.transporter_name[0]),                   
#         '1234',                    
#         10,                    
#         1
#         )

# prod_cursor.commit()
# conn.commit()

print("RRA Issued Completed..!")


Issuing Order :
userCode:69869 Delivery_Date:2026-03-12 00:00:00 route_id:01606 Office_Id:117
Issue For Dhavak: Shahid Ali Ahmed

RRA Issued Completed..!


# Sale Entry

In [56]:
grouped_orders = clean_order_booking_df.groupby(['kiosk_code','route_id_str', 'Select_Delivery_Date','userCode'])

for i in grouped_orders.groups:
    print(i)

(112407, '00700', Timestamp('2026-03-14 00:00:00'), 70872)


# Sale Entry Insertion

In [57]:
for key , order_df in grouped_orders:

    kiosk_code, route_id_str, delivery_date, userCode = key

    delivery_date = pd.to_datetime(delivery_date).to_pydatetime()

    print(f"\nInserting Order for Kiosk {kiosk_code}, Route {route_id_str}, Delivery Date {delivery_date}, User {userCode}")

    prod_cursor.execute("""
    EXEC drishteeDhavak..sp_scm_rrp_sale_booking_insert ?,?,?,?
                        """,
                        int(kiosk_code),
                        str(route_id_str),
                        delivery_date,
                        int(userCode)
                    )
    
    print("Inserted...!")
    
prod_cursor.commit()
conn.commit()

print("Sale Entry Completed..!")
    


Inserting Order for Kiosk 112407, Route 00700, Delivery Date 2026-03-14 00:00:00, User 70872
Inserted...!
Sale Entry Completed..!
